# Data Cleaning & Visualization — Predictive Maintenance

Engineering dataset: AI4I 2020 Predictive Maintenance Dataset.

Source: UCI Machine Learning Repository — https://archive.ics.uci.edu/dataset/601/ai4i+2020+predictive+maintenance+dataset

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

csv_path = Path('ai4i2020.csv')
zip_path = Path('ai4i2020.zip')
url = 'https://archive.ics.uci.edu/static/public/601/ai4i+2020+predictive+maintenance+dataset.zip'

if not csv_path.exists():
    if not zip_path.exists():
        urlretrieve(url, zip_path)
    with ZipFile(zip_path) as z:
        z.extract('ai4i2020.csv')

df = pd.read_csv(csv_path)
df.columns = (df.columns.str.strip().str.lower().str.replace(' ', '_', regex=False)
              .str.replace('[', '', regex=False).str.replace(']', '', regex=False))
df.head()

## Initial inspection

The dataset is checked for size, data types, missing values, duplicates, and numerical ranges before cleaning decisions are made.

In [ ]:
print('Shape:', df.shape)
df.info()
print('\nMissing values:')
print(df.isna().sum())
print('\nDuplicate rows:', df.duplicated().sum())
print('Duplicate UDI values:', df['udi'].duplicated().sum())
df.describe().T

## Cleaning

The dataset has no missing values, so imputation is not required. Duplicate rows and UDI values are absent, so no observations are removed for duplication. UDI and Product ID are identifiers rather than machine measurements and are excluded from the analytical dataframe.

In [ ]:
cleaned = df.drop(columns=['udi', 'product_id'])
sensor_cols = ['air_temperature_k', 'process_temperature_k', 'rotational_speed_rpm', 'torque_nm', 'tool_wear_min']
q1 = cleaned[sensor_cols].quantile(0.25)
q3 = cleaned[sensor_cols].quantile(0.75)
iqr = q3 - q1
outliers = ((cleaned[sensor_cols] < q1 - 1.5 * iqr) | (cleaned[sensor_cols] > q3 + 1.5 * iqr)).sum()
print('Remaining missing values:', cleaned.isna().sum().sum())
print('IQR outlier counts:')
print(outliers.sort_values(ascending=False))

## Failure analysis

Machine failure is relatively rare, so failure rates are more informative than raw counts when comparing groups. Failure-mode columns can overlap because one observation can satisfy more than one failure condition.

In [ ]:
failure_rate = cleaned['machine_failure'].mean() * 100
print(f'Failures: {cleaned.machine_failure.sum()} of {len(cleaned)} ({failure_rate:.2f}%)')
print('\nFailure modes:')
print(cleaned[['twf', 'hdf', 'pwf', 'osf', 'rnf']].sum().sort_values(ascending=False))
print('\nFailure rate by product type (%):')
print(cleaned.groupby('type').machine_failure.mean().mul(100).round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.boxplot(data=cleaned, x='machine_failure', y='air_temperature_k', ax=axes[0])
sns.boxplot(data=cleaned, x='machine_failure', y='tool_wear_min', ax=axes[1])
axes[0].set(xlabel='Machine failure', ylabel='Air temperature (K)', title='Air temperature by failure')
axes[1].set(xlabel='Machine failure', ylabel='Tool wear (min)', title='Tool wear by failure')
plt.tight_layout()
plt.show()

In [ ]:
sns.scatterplot(data=cleaned, x='rotational_speed_rpm', y='torque_nm', hue='machine_failure', alpha=0.55)
plt.xlabel('Rotational speed (rpm)')
plt.ylabel('Torque (Nm)')
plt.title('Torque and rotational speed by machine failure')
plt.show()

In [ ]:
corr_cols = sensor_cols + ['machine_failure']
sns.heatmap(cleaned[corr_cols].corr(), annot=True, fmt='.2f', center=0)
plt.title('Correlation between operating variables and machine failure')
plt.tight_layout()
plt.show()

## Findings

- The dataset contains 10,000 machine observations and no missing values, so imputation is not required.
- Duplicate rows and duplicate UDI values are absent, so no observations are removed for duplication.
- There are 339 machine failures (3.39%), making failure a minority event in the dataset.
- Heat dissipation failure is the most frequent individual failure mode in the dataset; failure-mode counts can overlap.
- Rotational speed and torque have a strong inverse correlation of about -0.88.
- Torque has the strongest direct correlation with machine failure among the operating variables shown, at about 0.19, while tool wear has a weaker positive correlation of about 0.11.
- Statistically unusual sensor readings identified by the IQR method are retained because an unusual measurement is not automatically an erroneous measurement.
- Identifier columns are removed from the analytical dataframe because they do not describe machine operating conditions.